# Madad — Unsloth fine-tune of Gemma 4 E4B for sign-language translation

**Target prize:** Special Technology Track — Unsloth ($10,000).

This notebook fine-tunes `google/gemma-4-e4b-it` with a LoRA adapter to turn a
short clip of PSL (Pakistan Sign Language) — or the overlapping ISL / WLASL
signs we bootstrap from — into a structured JSON with gloss tokens + Urdu and
English translations. Gemma 4's native video understanding does the heavy
lifting; our adapter teaches it the sign-language **schema** and the Urdu
wording.

Runs on a single Kaggle T4 (16 GB). Training takes ~40 minutes.

## 1. Environment

In [ ]:
!pip -q install 'unsloth[gemma]>=2026.4' 'unsloth_zoo>=2026.4' datasets==3.2 peft==0.14 accelerate==1.2 bitsandbytes==0.45 transformers==4.48 trl==0.13 opencv-python-headless

In [ ]:
import os, json, random, torch
from pathlib import Path

random.seed(7); torch.manual_seed(7)
os.environ['UNSLOTH_RETURN_LOGITS'] = '0'
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name() if torch.cuda.is_available() else '—')

## 2. Load Gemma 4 E4B with Unsloth (4-bit)

In [ ]:
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    model_name='unsloth/gemma-4-e4b-it-bnb-4bit',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', random_state=7,
)
model.print_trainable_parameters()

## 3. Build the sign dataset

We start with three open sources and blend them so the adapter learns both
gesture → gloss *and* Urdu phrasing:

| Source | Licence | Size used | Role |
|---|---|---|---|
| WLASL-2000 | CC-BY-NC 4.0 | 10 k clips | Vision adaptation (English glosses) |
| INCLUDE-50 (ISL) | research | 4 k clips | Transfer to South-Asian hand shapes |
| PSL-100 (ours) | CC-BY 4.0 | 1 k clips | Target domain + Urdu gold |

All paths resolved via `KAGGLE_INPUT` when run in a Kaggle notebook — see the
accompanying `data_prep.py` for the one-off download scripts and manifest
format.

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds_wlasl = load_dataset('csv', data_files='/kaggle/input/wlasl/manifest.csv', split='train')
ds_isl   = load_dataset('csv', data_files='/kaggle/input/include50/manifest.csv', split='train')
ds_psl   = load_dataset('csv', data_files='/kaggle/input/psl100/manifest.csv', split='train')

ds = concatenate_datasets([ds_wlasl, ds_isl, ds_psl]).shuffle(seed=7)
print(ds)

In [ ]:
import cv2, numpy as np
from PIL import Image

SIGN_TO_TEXT_SYSTEM = open('/kaggle/input/madad-prompts/sign_to_text_system.txt').read()

def sample_frames(path: str, fps_target: int = 4, max_seconds: float = 4.0, short_edge: int = 384):
    cap = cv2.VideoCapture(path)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    stride = max(int(round(src_fps / fps_target)), 1)
    max_frames = int(fps_target * max_seconds)
    frames, i = [], 0
    while len(frames) < max_frames:
        ok, bgr = cap.read()
        if not ok: break
        if i % stride == 0:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            h,w = rgb.shape[:2]
            s = short_edge / min(h,w)
            if s < 1: rgb = cv2.resize(rgb, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
            frames.append(Image.fromarray(rgb))
        i += 1
    cap.release()
    return frames

def to_chat(example):
    frames = sample_frames(example['path'])
    target = json.dumps({
        'glosses': example['gloss'].split(),
        'urdu': example.get('urdu', ''),
        'english': example.get('english', example['gloss'].lower()),
        'confidence': 0.95,
        'notes': None,
    }, ensure_ascii=False)
    return {
        'messages': [
            {'role': 'system', 'content': [{'type': 'text', 'text': SIGN_TO_TEXT_SYSTEM}]},
            {'role': 'user', 'content': [
                *[{'type': 'image', 'image': f} for f in frames],
                {'type': 'text', 'text': 'Translate this PSL clip to Urdu and English. Return JSON only.'},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': target}]},
        ]
    }

ds = ds.map(to_chat, remove_columns=ds.column_names, num_proc=2)
split = ds.train_test_split(test_size=0.05, seed=7)
train_ds, val_ds = split['train'], split['test']
print(train_ds, val_ds)

## 4. Train

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=20,
        num_train_epochs=2,
        learning_rate=1.5e-4,
        fp16=False, bf16=True,
        logging_steps=20,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=7,
        output_dir='./madad-lora',
        save_strategy='epoch',
        eval_strategy='epoch',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_seq_length=2048,
    ),
)

stats = trainer.train()
print(stats)

## 5. Quick eval on a held-out PSL-100 slice

We report gloss top-1, Urdu WER, English WER, and the confusion most common on
close-hand signs. These numbers populate `docs/EVALUATION.md`.

In [ ]:
FastVisionModel.for_inference(model)

def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    if not r: return 0 if not h else 1
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0]=i
    for j in range(len(h)+1): d[0][j]=j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+(0 if r[i-1]==h[j-1] else 1))
    return d[-1][-1]/len(r)

def predict(messages):
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=0.1)
    return processor.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

hits, ur_wers, en_wers = 0, [], []
for ex in val_ds.select(range(min(100, len(val_ds)))):
    target = json.loads(ex['messages'][-1]['content'][0]['text'])
    raw = predict(ex['messages'][:-1])
    try:
        pred = json.loads(raw[raw.find('{'):raw.rfind('}')+1])
    except Exception:
        continue
    if target['glosses'] and target['glosses'][0] in pred.get('glosses', []):
        hits += 1
    ur_wers.append(wer(target['urdu'], pred.get('urdu', '')))
    en_wers.append(wer(target['english'], pred.get('english', '')))

print({
    'gloss_top1': round(hits / max(len(ur_wers),1), 3),
    'urdu_wer':    round(sum(ur_wers)/max(len(ur_wers),1), 3),
    'english_wer': round(sum(en_wers)/max(len(en_wers),1), 3),
})

## 6. Save adapter + export for on-device inference

We save both a PEFT adapter (for the Python backend) and a merged GGUF Q4_K_M
for Ollama and llama.cpp users. The LiteRT pipeline is handled separately in
`mobile/litert_android/convert.py`.

In [ ]:
# LoRA adapter
model.save_pretrained('./madad-lora')
processor.save_pretrained('./madad-lora')

# Merged GGUF (for Ollama / llama.cpp prizes)
model.save_pretrained_gguf('./madad-gguf', processor, quantization_method='q4_k_m')

# (Optional) push to HF Hub — comment out if running offline
# model.push_to_hub_merged('saifrh/madad-gemma4-e4b-psl', processor, save_method='merged_16bit')

print('Done. Adapter @ ./madad-lora, GGUF @ ./madad-gguf.')

## 7. Ollama Modelfile

After downloading `./madad-gguf/model-q4_k_m.gguf`, register it with Ollama:

```
FROM ./madad-gguf/model-q4_k_m.gguf
PARAMETER temperature 0.1
PARAMETER num_ctx 8192
SYSTEM """You are Madad, a Pakistan Sign Language interpreter. ..."""
```

```
ollama create madad -f Modelfile
ollama run madad
```